# Expertise: from data entry to RDF

This notebook shows how to describe a person's areas of expertise and convert
that description into a standardised, machine-readable RDF graph.

**You only need to edit one file:** `docs/example.oold.json`.  Everything
else is automatic.

---

## What the notebook does

```
example.oold.json
  │  URIs for each area of expertise
  │
  ▼
RDF graph
  │  machine-readable, ontology-aligned triples
  │
  ▼
(no SHACL shape yet; structural correctness enforced by JSON Schema)
```

> **No transform step:** the expertise input is already in OO-LD format;
> the fields map directly to ontology IRIs.  There is nothing to pre-process.

---

## Environment setup

```bash
python3 -m venv .venv
source .venv/bin/activate
pip install jupyterlab
jupyter lab
```

In [ ]:
%pip install -q rdflib pyyaml

In [ ]:
import sys
import json
import pathlib
import yaml
from importlib.metadata import version
import rdflib

print("Python :", sys.executable)
print()
print("rdflib  ", version("rdflib"))

# Paths
HERE   = pathlib.Path().resolve()   # docs/
SCHEMA = HERE.parent                # expertise/

---
## Step 1: Describe a person's expertise

Edit `docs/example.oold.json` with your data, then run this cell to load it.

Each field is an array of knowledge-graph URIs, one per concept
(e.g. a material, a process, a measurement method) that this person
works with.  The available URIs come from your DSMS knowledge graph.

> **Tip:** you can omit any field whose array would be empty; leave it out
> entirely rather than setting it to `[]`.

In [ ]:
doc = json.loads((HERE / "example.oold.json").read_text())

print(json.dumps(doc, indent=2))

---
## Step 2: Convert to RDF

The OO-LD document is parsed as JSON-LD using the ontology context from
`specs/schema.oold.yaml`, which maps every field name to its ontology IRI.
rdflib ≥ 7 handles JSON-LD 1.1 natively.

In [ ]:
context = yaml.safe_load((SCHEMA / "specs" / "schema.oold.yaml").read_text())["@context"]

g = rdflib.Dataset()
g.parse(data=json.dumps({"@context": context, **doc}), format="json-ld")

print(f"Graph contains {len(g)} triples.\n")
print(g.serialize(format="turtle"))

In [ ]:
# Optional: save to file
out_ttl = HERE / "output_expertise.ttl"
out_ttl.write_text(g.serialize(format="turtle"))
print(f"Written to {out_ttl}")

---
## Summary

| Step | What happens |
|---|---|
| 1 | You fill in `example.oold.json` with the relevant knowledge-graph URIs |
| 2 | The OO-LD document is parsed into an RDF graph |

To describe a different expert, edit `example.oold.json` and re-run all cells.

---

## Further reading

- [OO-LD primer](../../../docs/oold-primer.md): what OO-LD is and how it works
- [Schema format reference](../../../docs/schema-format.md): how to write your own schema